In [1]:
pip install -U langchain langchain_openai langchain_community langchain_classic python-dotenv

Python(45236) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pymupdf pythainlp

Python(45238) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


get text from pdf

In [3]:
import fitz
from pythainlp.util import normalize

doc = fitz.open("../document/OEB3_425_MB_CS_Syllabus.pdf")

text = ""

for i in range(len(doc)):
    page = doc.load_page(i)
    t = page.get_text()
    t = normalize(t)
    t += f"\n หน้าที่ {i+1} \n"
    text += t

doc.close()
print(text)

หลักสูตรวิทยาศาสตรบัณฑิต สาขาวิชา วิทยาการคอมพิวเตอร:
ภาควิชา วิทยาการคอมพิวเตอร:และสารสนเทศ
ระดับปริญญาตรี คณะวิทยาศาสตร:ประยุกต:
Syllabus
1
รายวิชา
040613421 การพัฒนาโปรแกรมประยุกต:เคลื่อนที่
Mobile Application Development
ชื่อสถาบันอุดมศึกษา มหาวิทยาลัยเทคโนโลยีพระจอมเกล7าพระนครเหนือ
วิทยาเขต กรุงเทพมหานคร คณะ วิทยาศาสตร>ประยุกต> ภาควิชา วิทยาการคอมพิวเตอร>และสารสนเทศ
หมวดที่ 1 ขHอมูลทั่วไป
1. รหัสและชื่อรายวิชา
040613421 การพัฒนาโปรแกรมประยุกต>เคลื่อนที่ (Mobile Application Development)
2. จำนวนหนSวยกิต
3(2-2-5)
3. หลักสูตรและประเภทของรายวิชา
หลักสูตรวิทยาศาสตรบัณฑิต สาขาวิชา วิทยาการคอมพิวเตอร>
เปaนรายวิชา ชีพเลือก
4. อาจารยWผูHรับผิดชอบรายวิชาและอาจารยWผูHสอน
อาจารย>ผู7รับผิดชอบรายวิชาและอาจารย>ผู7สอน
รศ.ดร.กอบเกียรติ สระอุบล
5. รายวิชาบังคับกSอน (Pre-requisite) (ถHามี)
040613204 การโปรแกรมเชิงวัตถุ
หมวดที่ 2 ลักษณะและการดำเนินการ
1. คำอธิบายรายวิชา
สถาป%ตยกรรมระบบ สภาพแวดล4อมสาหรับการพัฒนาแบบรวม การพัฒนาส:วนต:อประสานผู4ใช4
การทดสอบและการแก4ไข ฐานข4อมูล กราฟDกสEและมัลติมีเดีย การ

split text

In [4]:
import os
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pythainlp.util import normalize 

doc_folder = "../document"
docs = []

# 1. วนลูปอ่านทุกไฟล์ในโฟลเดอร์
for filename in os.listdir(doc_folder):
    if filename.endswith(".pdf"):
        file_path = os.path.join(doc_folder, filename)
        doc = fitz.open(file_path)
        
        # 2. วนลูปอ่านทีละหน้าเหมือนที่คุณทำ
        for i, page in enumerate(doc):
            text = normalize(page.get_text())
            docs.append(Document(
                page_content=text, 
                metadata={"source": filename, "page": i+1}
            ))

# 3. สั่งแบ่ง Chunk ครั้งเดียวจบ (มันจะแบ่งตามเนื้อหาที่เราเก็บมา)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print(f"Total chunks: {len(chunks)} จากไฟล์ทั้งหมด")


/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 50 จากไฟล์ทั้งหมด


In [5]:
pip install -U sentence-transformers faiss-cpu

Python(45264) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-m3',encode_kwargs={"normalize_embeddings" : True})

vectorstore = FAISS.from_documents(chunks,embeddings)
retriever = vectorstore.as_retriever()

/var/folders/3d/grsq7_t974n5lybhnf0hr4qh0000gn/T/ipykernel_23366/2309049932.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-m3',encode_kwargs={"normalize_embeddings" : True})
Python(45278) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 40009.10it/s]


In [7]:
vectorstore.save_local(folder_path="faiss_index")

In [8]:
embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-m3',encode_kwargs={"normalize_embeddings" : True})

new_db = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = new_db.as_retriever()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 49801.79it/s]


In [9]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.memory import ConversationBufferMemory

load_dotenv()

retriever = new_db.as_retriever(search_kwargs={"k": 10}) 

# 1. เพิ่ม max_tokens และปรับอุณหภูมิให้ตอบธรรมชาติขึ้น
llm = ChatOpenAI(
    base_url="https://api.opentyphoon.ai/v1",
    api_key=os.getenv("TYPHOON_KEY"),
    model='typhoon-v2.5-30b-a3b-instruct',
    temperature=0.3,
    max_tokens=8192 # เพิ่มขึ้นนิดนึงเพื่อให้ภาษาเป็นธรรมชาติ  # ⭐️ เพิ่มความยาวสูงสุดของคำตอบ (ปกติจะอยู่ราวๆ 256-512)
)

# 2. ปรับ Prompt ให้เป็นบุคลิกแชทบอทใจดี แนะนำข้อมูลได้
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "คุณคือแชทบอทที่ปรึกษาด้านวิชาเรียน ชื่อ 'Learnify Bot' แสนเป็นมิตรและพร้อมช่วยเหลือ\n"
     "หน้าที่ของคุณคือ:\n"
     "1. ตอบคำถามเกี่ยวกับรายวิชาโดยอ้างอิงจาก 'ข้อมูลอ้างอิง' เท่านั้น\n"
     "2. หากมีข้อมูลที่สอดคล้องกับสิ่งที่ผู้ใช้สนใจ (แม้ผู้ใช้จะใช้ภาษาพูด เช่น 'ทำแอพ', 'เว็บ') ให้ดึงข้อมูลมาตอบได้เลย\n"
     "3. บอก รหัสวิชา, ชื่อวิชา (ไทยและอังกฤษ) และอาจเสริมคำอธิบายสั้นๆ ว่าวิชานี้เกี่ยวกับการทำสิ่งนั้นได้อย่างไร เพื่อให้ผู้ใช้เห็นภาพ\n"
     "4. ตอบด้วยความสุภาพ เป็นกันเอง มีการจัดเรียงข้อความให้อ่านง่าย\n"
     "5. หากสืบค้นแล้วไม่พบข้อมูล ให้ตอบอย่างสุภาพว่าไม่พบรายวิชาที่ตรงกับความต้องการในขณะนี้"),
    
    ("human", "ข้อมูลอ้างอิง:\n{context}\n\nคำถาม: {question}")
])

parser = StrOutputParser()

chain = ({"context": retriever, "question": RunnablePassthrough()} | prompt | llm | parser)

# 3. ลองทดสอบคุย
res = chain.invoke("Mobile developer วิชานี้ตัดเกรดอย่างไรสอบกลางภาคคิดกี่เปอร์เซ็นต์มีการทำโครงงานหรือไม่ผู้สอนมีใครบ้างวิชาเกี่ยวกับอะไรมีรายวิชาบังคับก่อนไหม (Pre-requisite)")

print(res)


สวัสดีค่ะ Learnify Bot ยินดีช่วยเสมอเลย 😊  
จากข้อมูลที่คุณถามมา เรามาดูรายละเอียดของ **รายวิชา "040613421 การพัฒนาโปรแกรมประยุกต์เคลื่อนที่"** (Mobile Application Development) กันนะคะ

---

### 📚 รหัสวิชา & ชื่อวิชา
- **รหัสวิชา:** 040613421  
- **ชื่อวิชา (ไทย):** การพัฒนาโปรแกรมประยุกต์เคลื่อนที่  
- **ชื่อวิชา (อังกฤษ):** Mobile Application Development  

> ✅ วิชานี้เหมาะกับคนที่อยากเป็น **Mobile Developer** เพราะเน้นการพัฒนาแอปพลิเคชันบนมือถือจริง ๆ ทั้งในด้านสถาปัตยกรรม ฟีเจอร์ และการทดสอบการใช้งาน

---

### 🎯 ผลลัพธ์การเรียนรู้ (CLOs)
วิชานี้เน้นให้นักศึกษาสามารถ:
- ออกแบบและพัฒนาแอปพลิเคชันเคลื่อนที่ได้
- เข้าใจสถาปัตยกรรมของแอปพลิเคชันมือถือ
- ใช้เครื่องมือและเทคโนโลยีที่เกี่ยวข้องในการพัฒนา

---

### 🧾 การตัดเกรด (เกณฑ์การประเมินผล)

| กิจกรรม | สัดส่วน | กำหนดการ (สัปดาห์ที่) |
|--------|--------|----------------|
| สอบกลางภาค | **30%** | สัปดาห์ที่ 8 |
| สอบปลายภาค | **30%** | สัปดาห์ที่ 16 |
| โครงงาน | **10%** | สัปดาห์ที่ 16 |
| การเข้าเรียน / งานที่มอบหมาย / การปฏิบัติ 